In [1]:
import torch
import numpy as np
import matplotlib.pyplot as plt 
import os
import sys
sys.path.append(os.path.abspath(os.path.join('..')))
from megatron.checkpointing import model_diff, sparsification, csr_sparsification

Zarr-based strategies will not be registered because of missing packages
/home/pengyanxin/.local/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
ckpt_path2 = "/mnt/pengyanxin/my_megatron/examples/gpt3/gpt2_345m/iter_0002228/mp_rank_00/model_optim_rng.pt"
ckpt_path1 = "/mnt/pengyanxin/my_megatron/examples/gpt3/gpt2_345m/iter_0002227/mp_rank_00/model_optim_rng.pt"
ckpt1 = torch.load(ckpt_path1, map_location='cpu')
ckpt2 = torch.load(ckpt_path2, map_location='cpu')


model_state_dict1 = ckpt1['model']
model_state_dict2 = ckpt2['model']


In [3]:
print(model_state_dict1['language_model']['embedding'])


{'word_embeddings': OrderedDict([('weight', tensor([[ 0.0286,  0.0401,  0.0729,  ..., -0.0939, -0.1409, -0.1028],
        [ 0.0176,  0.0257,  0.0098,  ..., -0.0256, -0.0937, -0.0145],
        [ 0.0171,  0.0461,  0.0047,  ..., -0.0517, -0.0982, -0.0508],
        ...,
        [-0.0211,  0.0131, -0.0081,  ...,  0.0246,  0.0217, -0.0002],
        [-0.0258, -0.0025, -0.0046,  ...,  0.0126,  0.0179, -0.0248],
        [-0.0424,  0.0187, -0.0275,  ...,  0.0444,  0.0149, -0.0299]],
       dtype=torch.float16))]), 'position_embeddings': OrderedDict([('weight', tensor([[ 0.0102,  0.0214, -0.0269,  ...,  0.0066,  0.0047,  0.0060],
        [ 0.0405, -0.0018,  0.0303,  ..., -0.0063, -0.0099,  0.0149],
        [ 0.0155,  0.0091,  0.0128,  ...,  0.0062, -0.0205,  0.0243],
        ...,
        [-0.0185, -0.0026,  0.0221,  ..., -0.0252, -0.0172, -0.0126],
        [-0.0055,  0.0153,  0.0188,  ..., -0.0225, -0.0196, -0.0104],
        [-0.0114, -0.0035,  0.0219,  ..., -0.0153, -0.0112, -0.0142]],
       dt

In [4]:

diff = {}
model_diff(model_state_dict1, model_state_dict2, diff)

print(diff)

{'language_model': {'embedding': {'word_embeddings': {'weight': tensor([[ 3.0518e-05,  6.1035e-05, -6.1035e-05,  ...,  6.1035e-05,
          1.2207e-04,  0.0000e+00],
        [-3.0518e-05, -1.5259e-05,  3.0518e-05,  ..., -1.0681e-04,
         -6.1035e-05,  1.5259e-05],
        [-3.0518e-05, -3.0518e-05, -7.6294e-06,  ..., -3.0518e-05,
          0.0000e+00,  3.0518e-05],
        ...,
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00, -1.1921e-07],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00],
        [ 0.0000e+00,  0.0000e+00,  0.0000e+00,  ...,  0.0000e+00,
          0.0000e+00,  0.0000e+00]], dtype=torch.float16)}, 'position_embeddings': {'weight': tensor([[-3.0518e-05, -3.0518e-05, -4.5776e-05,  ..., -3.0518e-05,
         -3.4332e-05, -4.1962e-05],
        [-3.0518e-05,  5.4359e-05, -3.0518e-05,  ...,  7.6294e-06,
         -7.6294e-06, -3.0518e-05],
        [ 3.8147e-05,  1.5259e-05,  0.000

In [5]:
def generate_bitmask(delta, bitmasks):
    for key, value in delta.items():
        if isinstance(value, dict):
            bitmasks[key] = {}
            generate_bitmask(value, bitmasks[key])
        elif isinstance(value, torch.Tensor):
            bitmasks[key] = (value != 0).to(torch.uint8)  # Use uint8 instead of bool
        elif value is None:
            bitmasks[key] = None
        else:
            raise ValueError(f"Unsupported type for key '{key}': {type(value)}")

In [7]:
def flatten_tensor(tensor):
    return tensor.view(-1)

def store_sparse_delta_with_bitmask(delta, bitmask, file_path):
    def process_delta_and_bitmask(delta, bitmask):
        sparse_delta_with_bitmask = {}
        for key in delta.keys():
            if isinstance(delta[key], dict):
                sparse_delta_with_bitmask[key] = process_delta_and_bitmask(delta[key], bitmask[key])
            elif isinstance(delta[key], torch.Tensor):
                flattened_delta = flatten_tensor(delta[key])
                flattened_bitmask = flatten_tensor(bitmask[key])
                sparse_values = flattened_delta[flattened_bitmask == 1]
                sparse_delta_with_bitmask[key] = (sparse_values, flattened_bitmask)
            elif delta[key] is None:
                sparse_delta_with_bitmask[key] = None
            else:
                raise ValueError(f"Unsupported type for key '{key}': {type(delta[key])}")
        return sparse_delta_with_bitmask

    sparse_delta_with_bitmask = process_delta_and_bitmask(delta, bitmask)
    torch.save(sparse_delta_with_bitmask, file_path)

def load_sparse_delta_with_bitmask(file_path):
    return torch.load(file_path)

# Store the sparse delta and bitmask
bitmasks = {}
generate_bitmask(diff, bitmasks)
store_sparse_delta_with_bitmask(diff, bitmasks, 'sparse_delta.pt')

In [ ]:
def load_sparse_delta_with_bitmask(file_path):
    return torch.load(file_path)

def reconstruct_checkpoint_with_bitmask(base_checkpoint, sparse_delta_with_bitmask):
    def process_reconstruction(base_checkpoint, sparse_delta_with_bitmask):
        new_checkpoint = {}
        for key in base_checkpoint.keys():
            if key in sparse_delta_with_bitmask:
                if isinstance(sparse_delta_with_bitmask[key], tuple):
                    sparse_values, flattened_bitmask = sparse_delta_with_bitmask[key]
                    delta = torch.zeros_like(base_checkpoint[key]).view(-1)
                    delta[flattened_bitmask == 1] = sparse_values
                    delta = delta.view(base_checkpoint[key].shape)
                    new_checkpoint[key] = base_checkpoint[key] + delta
                elif isinstance(sparse_delta_with_bitmask[key], dict):
                    new_checkpoint[key] = process_reconstruction(base_checkpoint[key], sparse_delta_with_bitmask[key])
                elif sparse_delta_with_bitmask[key] is None:
                    new_checkpoint[key] = base_checkpoint[key]
                else:
                    raise ValueError(f"Unsupported type for key '{key}' in sparse_delta_with_bitmask: {type(sparse_delta_with_bitmask[key])}")
            else:
                new_checkpoint[key] = base_checkpoint[key]
        return new_checkpoint

    return process_reconstruction(base_checkpoint, sparse_delta_with_bitmask)

recovered_state_dict = reconstruct_checkpoint_with_bitmask(model_state_dict1, load_sparse_delta_with_bitmask('sparse_delta.pt'))
print(recovered_state_dict)





In [ ]:
test = {}
model_diff(model_state_dict2, recovered_state_dict, test)
print(test)

In [ ]:
print(bitmasks)

In [ ]:
sparse_model_dict = {}
sparsification(diff, sparse_model_dict)



In [ ]:
print(sparse_model_dict)


In [ ]:
torch.save(sparse_model_dict, "sparse_model.pt")

In [ ]:
def tensors_to_numpy(data):
    if isinstance(data, dict):
        return {key: tensors_to_numpy(value) for key, value in data.items()}
    elif isinstance(data, torch.Tensor):
        return data.cpu().numpy()
    else:
        return data

def calculate_percentage_of_changes(tensor_diff):
    # Flatten the tensor to a 1D array
    if tensor_diff is None:
        return 0.0
    
    tensor_diff_flat = tensor_diff.flatten()
    
    # Count the number of non-zero elements
    num_non_zero = (tensor_diff_flat != 0).sum()
    
    # Calculate the total number of elements
    total_elements = tensor_diff_flat.size
    
    # Compute the percentage of changed elements
    percentage_changed = (num_non_zero / total_elements) * 100
    
    return percentage_changed

diff_state_dict_numpy = tensors_to_numpy(diff)

total_elements = 0 
total_percentage = 0
for key in diff_state_dict_numpy['language_model']['encoder'].keys():
    tensor_diff_to_plot = diff_state_dict_numpy['language_model']['encoder'][key]
    percentage_changed = calculate_percentage_of_changes(tensor_diff_to_plot)
    print(f"Percentage of changed elements for key '{key}': {percentage_changed:.2f}%")
    total_percentage += percentage_changed
    total_elements += 1

print(f"Average percentage of changed elements: {total_percentage / total_elements:.2f}%")

In [ ]:
diff_state_dict = {}
def iterate_and_subtract(dict1, dict2, diff_dict):
    for key in dict1.keys(): 
        if key in dict2.keys():
            if isinstance(dict1[key], dict) and isinstance(dict2[key], dict):
                diff_dict[key] = {}
                iterate_and_subtract(dict1[key], dict2[key], diff_dict[key])
            elif isinstance(dict1[key], torch.Tensor) and isinstance(dict2[key], torch.Tensor):
                diff_dict[key] = dict1[key] - dict2[key]
                max_diff_index = torch.argmax(torch.abs(diff_dict[key]))
                max_diff_indices = torch.unravel_index(max_diff_index, diff_dict[key].shape)
                dict1_value = dict1[key][max_diff_indices]
                dict2_value = dict2[key][max_diff_indices]
                max_diff_value = diff_dict[key][max_diff_indices]
                print(f"Max diff for key '{max_diff_indices}': {max_diff_value}")
                print(f"Value in dict1 for key '{max_diff_indices}: {dict1_value}")
                print(f"Value in dict1 for key '{max_diff_indices}: {dict2_value}")
            else:
                raise ValueError(f"Mismatched types for key '{key}': {type(dict1[key])} vs {type(dict2[key])}")
        else:
            raise ValueError(f"Key '{key}' not found in second dict")

iterate_and_subtract(state_dict1, state_dict2, diff_state_dict)

print(diff_state_dict['language_model']['encoder'].keys())

print(diff_state_dict)

In [ ]:
def tensors_to_numpy(data):
    if isinstance(data, dict): 
        return {key: tensors_to_numpy(value) for key, value in data.items()}
    elif isinstance(data, torch.Tensor): 
        if data.dtype == torch.bfloat16:
            data = data.to(torch.float16)
        return data.cpu().numpy()
    else: 
        return data
    
diff_state_dict_numpy = tensors_to_numpy(diff_state_dict)

In [ ]:
def plot_scatter_for_row_or_column(tensor_diff, index=0, axis=0, title="Scatter Plot of Differences", max_dots=1000):
    if axis == 0:
        data = tensor_diff[index, :]
    else:
        data = tensor_diff[:, index]
    
    data = np.array(data)
    
    plt.figure(figsize=(10, 6))
    plt.scatter(range(len(data[:max_dots])), data[:max_dots], alpha=0.6)
    plt.title(title)
    plt.xlabel("Index")
    plt.ylabel("Value")
    plt.show()
diff_tensor = diff_state_dict_numpy['language_model']['encoder']['layers.0.mlp.dense_h_to_4h.weight'] 
plot_scatter_for_row_or_column(diff_tensor, index=3525, axis=0, title="Scatter Plot of Row Differences", max_dots=1000)

In [ ]:
def plot_non_zero_heatmap(tensor_diff, title="Non-zero Differences Heatmap"):
    # Create a mask for non-zero values
    non_zero_mask = tensor_diff != 0
    
    # Plot the heatmap
    plt.figure(figsize=(10, 8))
    sns.heatmap(non_zero_mask, cbar=False, cmap='viridis')
    plt.title(title)
    plt.xlabel("D1: rows of the weight matrix")
    plt.ylabel("D2: columns of the weight matrix")
    plt.show()



In [ ]:
# Example of accessing and plotting a specific tensor difference
key_to_plot = 'language_model.encoder.layers.0.self_attention.query_key_value.weight'
tensor_diff_to_plot = diff_state_dict_numpy['language_model']['encoder']['layers.0.mlp.dense_h_to_4h.weight']

# Plotting the heatmap of non-zero differences
plot_non_zero_heatmap(tensor_diff_to_plot, title=key_to_plot)


In [ ]:
print(tensor_diff_to_plot.shape)

In [ ]:
def calculate_percentage_of_changes(tensor_diff):
    # Flatten the tensor to a 1D array
    tensor_diff_flat = tensor_diff.flatten()
    
    # Count the number of non-zero elements
    num_non_zero = (tensor_diff_flat != 0).sum()
    
    # Calculate the total number of elements
    total_elements = tensor_diff_flat.size
    
    # Compute the percentage of changed elements
    percentage_changed = (num_non_zero / total_elements) * 100
    
    return percentage_changed

# Example usage
for key in diff_state_dict_numpy['language_model']['encoder'].keys():
    tensor_diff_to_plot = diff_state_dict_numpy['language_model']['encoder'][key]
    percentage_changed = calculate_percentage_of_changes(tensor_diff_to_plot)
    print(f"Percentage of changed elements for key '{key}': {percentage_changed:.2f}%")



In [ ]:
def tensors_to_numpy(data):
    if isinstance(data, dict): 
        return {key: tensors_to_numpy(value) for key, value in data.items()}
    elif isinstance(data, torch.Tensor): 
        if data.dtype == torch.bfloat16:
            data = data.to(torch.float16)
        return data.cpu().numpy()
    else: 
        return data
    
diff_state_dict_numpy = tensors_to_numpy(diff_state_dict)

In [ ]:
def find_max_change(diff_dict):
    max_change = 0
    max_change_key = None

    def recursive_find_max_change(d):
        nonlocal max_change, max_change_key
        for key, value in d.items():
            if isinstance(value, dict):
                recursive_find_max_change(value)
            elif isinstance(value, np.ndarray):
                current_max_change = np.max(np.abs(value))
                if current_max_change > max_change:
                    max_change = current_max_change
                    max_change_key = key
            else:
                raise ValueError(f"Unsupported type for key '{key}': {type(value)}")

    recursive_find_max_change(diff_dict)
    return max_change, max_change_key

# Example usage
max_change, max_change_key = find_max_change(diff_state_dict_numpy)
print(f"Maximum change: {max_change} in key: {max_change_key}")
print(diff_state_dict_numpy)